# Hiver SDE Intern Take-Home Assignment
## AmazonHelp AI Customer Support Agent

### Objective

Build an AI customer-support agent using real customer-support conversations that can:

1. Classify incoming customer messages into data-derived intents.
2. Retrieve relevant historical resolutions and draft a grounded response.
3. Decide whether the interaction can be auto-handled or should be escalated to a human.

### Selected Brand

AmazonHelp

### Final Evaluation

- Golden set: 200 manually labeled examples
- Intent OOF accuracy: 43.5%
- Intent weighted F1: 39.54%
- Leakage-safe retrieval success: 98.5%
- Escalation accuracy: 69.5%
- Escalation precision: 83.0%
- Escalation recall: 42.4%
- Escalation F1: 56.12%
- Auto-handle rate: 76.5%
- Human reply quality: 4.31/5

### Important Evaluation Note

Initial evaluation results were not used as the final headline numbers because they contained evaluation leakage. Intent classification was therefore evaluated using 5-fold out-of-fold predictions, and retrieval evaluation excluded the corresponding golden examples from the retrieval corpus.

# 1. Problem Statement

# 2. Environment Setup & Dataset Loading

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Path to dataset files: /kaggle/input/customer-support-on-twitter


In [2]:
import pandas as pd
import numpy as np
import os

file_path = os.path.join(path, "twcs", "twcs.csv")

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Dataset not found at: {file_path}"
    )

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (2811774, 7)
Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Missing values:
tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64

First 5 rows:


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


# 3. Brand Selection

In [61]:
import pandas as pd
import os
import kagglehub

# Ensure `path` is defined for dataset access
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
file_path = os.path.join(path, "twcs", "twcs.csv")

# Reload the original df to ensure it contains the 'inbound' column
df = pd.read_csv(file_path)

# Support tweets are outbound tweets: inbound == False
support_df = df[df["inbound"] == False].copy()

# Count tweets by support account
brand_counts = (
    support_df["author_id"]
    .value_counts()
    .reset_index()
)

brand_counts.columns = ["brand", "support_tweets"]

print("Number of support brands:", len(brand_counts))

print("\nTop 30 brands by support tweet volume:")
display(brand_counts.head(30))

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Number of support brands: 108

Top 30 brands by support tweet volume:


,brand,support_tweets
0,AmazonHelp,169840
1,AppleSupport,106860
2,Uber_Support,56270
3,SpotifyCares,43265
4,Delta,42253
5,Tesco,38573
6,AmericanAir,36764
7,TMobileHelp,34317
8,comcastcares,33031
9,British_Airways,29361


In [4]:
# PHASE 2 — COMPARE ALL BRANDS

import pandas as pd

# Support tweets
support = df[df["inbound"] == False].copy()

# Customer tweets
customer = df[df["inbound"] == True].copy()

# Convert IDs to strings to avoid merge type problems
support["tweet_id"] = support["tweet_id"].astype(str)
support["in_response_to_tweet_id"] = (
    support["in_response_to_tweet_id"]
    .astype("Int64")
    .astype(str)
)

customer["tweet_id"] = customer["tweet_id"].astype(str)

# Keep only support tweets that directly respond to a customer tweet
pairs = support[
    support["in_response_to_tweet_id"].notna()
].copy()

# Match support responses to customer tweets
pairs = pairs.merge(
    customer[["tweet_id", "text"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    how="inner",
    suffixes=("_support", "_customer")
)

# Brand-level statistics
brand_comparison = (
    pairs.groupby("author_id")
    .agg(
        support_responses=("tweet_id_support", "count"),
        unique_customers=("tweet_id_customer", "nunique"),
        unique_customer_messages=("text_customer", "nunique"),
        avg_customer_length=("text_customer", lambda x: x.str.len().mean()),
        avg_support_length=("text_support", lambda x: x.str.len().mean())
    )
    .reset_index()
)

brand_comparison = brand_comparison.rename(
    columns={"author_id": "brand"}
)

# Approximate conversation volume:
# each customer -> support pair is treated as one support interaction
brand_comparison["conversation_volume"] = (
    brand_comparison["support_responses"]
)

# Rank brands
brand_comparison = brand_comparison.sort_values(
    "conversation_volume",
    ascending=False
).reset_index(drop=True)

print("Brands with usable customer → support interactions:",
      len(brand_comparison))

display(brand_comparison)

Brands with usable customer → support interactions: 108


,brand,support_responses,unique_customers,unique_customer_messages,avg_customer_length,avg_support_length,conversation_volume
0,AmazonHelp,168814,154976,153004,116.589033,123.936925,168814
1,AppleSupport,106646,106623,104823,109.261379,136.622977,106646
2,Uber_Support,56160,55182,54491,120.221011,110.079291,56160
3,SpotifyCares,43092,41585,41050,103.937622,129.533742,43092
4,Delta,42114,36134,35595,110.702593,103.934535,42114
...,...,...,...,...,...,...,...
103,JackBox,255,252,252,97.180392,54.505882,255
104,OfficeSupport,211,188,187,120.729858,115.090047,211
105,AskDSC,210,210,210,111.461905,91.838095,210
106,CarlsJr,181,180,180,92.458564,110.116022,181


In [5]:
# PHASE 2 — AMAZONHELP VERIFICATION

amazon_stats = brand_comparison[
    brand_comparison["brand"].astype(str).str.lower() == "amazonhelp"
]

display(amazon_stats)

,brand,support_responses,unique_customers,unique_customer_messages,avg_customer_length,avg_support_length,conversation_volume
0,AmazonHelp,168814,154976,153004,116.589033,123.936925,168814


In [6]:
# PHASE 3 — BUILD CLEAN AMAZONHELP CONVERSATION PAIRS

import pandas as pd

# Select AmazonHelp support tweets
amazon_support = df[
    (df["inbound"] == False) &
    (df["author_id"].astype(str).str.lower() == "amazonhelp")
].copy()

# Select customer tweets
customer_tweets = df[
    df["inbound"] == True
].copy()

# Keep only support tweets that directly reply to another tweet
amazon_support = amazon_support[
    amazon_support["in_response_to_tweet_id"].notna()
].copy()

# Convert IDs to strings AFTER removing missing values
amazon_support["support_tweet_id"] = (
    amazon_support["tweet_id"]
    .astype(str)
)

amazon_support["customer_tweet_id"] = (
    amazon_support["in_response_to_tweet_id"]
    .astype("int64")
    .astype(str)
)

customer_tweets["customer_tweet_id"] = (
    customer_tweets["tweet_id"]
    .astype(str)
)

# Keep required columns
support_clean = amazon_support[
    [
        "support_tweet_id",
        "customer_tweet_id",
        "created_at",
        "text"
    ]
].rename(columns={
    "created_at": "support_time",
    "text": "support_text"
})

customer_clean = customer_tweets[
    [
        "customer_tweet_id",
        "created_at",
        "text"
    ]
].rename(columns={
    "created_at": "customer_time",
    "text": "customer_text"
})

# Create customer → AmazonHelp pairs
pairs_df = customer_clean.merge(
    support_clean,
    on="customer_tweet_id",
    how="inner"
)

# Remove empty messages
pairs_df = pairs_df.dropna(
    subset=["customer_text", "support_text"]
).copy()

# Remove duplicate conversation pairs
pairs_df = pairs_df.drop_duplicates(
    subset=["customer_tweet_id", "support_tweet_id"]
).reset_index(drop=True)

# Basic text cleaning
pairs_df["customer_text"] = (
    pairs_df["customer_text"]
    .astype(str)
    .str.strip()
)

pairs_df["support_text"] = (
    pairs_df["support_text"]
    .astype(str)
    .str.strip()
)

# Remove extremely short messages
pairs_df = pairs_df[
    (pairs_df["customer_text"].str.len() >= 5) &
    (pairs_df["support_text"].str.len() >= 5)
].reset_index(drop=True)

# Save
pairs_df.to_csv(
    "/content/amazonhelp_conversation_pairs.csv",
    index=False
)

print("AmazonHelp support tweets:", len(amazon_support))
print("Usable customer → support pairs:", len(pairs_df))

print("\nColumns:")
print(pairs_df.columns.tolist())

print("\nSample conversations:")
display(
    pairs_df[
        [
            "customer_text",
            "support_text"
        ]
    ].head(10)
)

AmazonHelp support tweets: 169287
Usable customer → support pairs: 168814

Columns:
['customer_tweet_id', 'customer_time', 'customer_text', 'support_tweet_id', 'support_time', 'support_text']

Sample conversations:


,customer_text,support_text
0,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
1,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
2,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
3,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
5,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
6,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
7,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
8,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
9,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM


PHASE 4 — Intent Discovery

In [7]:
# PHASE 4 — INTENT DISCOVERY FROM AMAZONHELP DATA

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Load the clean conversation dataset
pairs_df = pd.read_csv(
    "/content/amazonhelp_conversation_pairs.csv"
)

# Sample conversations for efficient clustering
sample_size = min(5000, len(pairs_df))

sample_df = pairs_df.sample(
    n=sample_size,
    random_state=2026
).reset_index(drop=True)

texts = sample_df["customer_text"].astype(str)

# TF-IDF representation
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3
)

X = tfidf.fit_transform(texts)

# Exploratory clustering
n_clusters = 10

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=2026,
    n_init=10
)

sample_df["cluster"] = kmeans.fit_predict(X)

# Display important terms and representative examples
terms = np.array(tfidf.get_feature_names_out())

print("Sample conversations:", len(sample_df))
print("TF-IDF features:", X.shape[1])
print("Clusters:", n_clusters)

for cluster_id in range(n_clusters):

    center = kmeans.cluster_centers_[cluster_id]

    top_indices = center.argsort()[-10:][::-1]

    top_terms = terms[top_indices]

    print(f"\nCluster {cluster_id}")
    print("Top terms:", ", ".join(top_terms))

    examples = sample_df[
        sample_df["cluster"] == cluster_id
    ]["customer_text"].head(5)

    for example in examples:
        print("-", example[:250])

Sample conversations: 5000
TF-IDF features: 4192
Clusters: 10

Cluster 0
Top terms: amazonhelp, https, 115850, just, yes, help, amazonhelp yes, 115830, thanks, email
- @AmazonHelp Filled out and sent.
- @AmazonHelp Yes, I’ve just been in contact with my bank and they said there are 2 pending transactions with yourselves neither of the are for the amount of the order put through
- @117795 suggested resolution was take it back to the post office even after explaining the whole reason for me using your service...
- @AmazonHelp Ya llego :) https://t.co/Ivpk9C3duM
- Non è che per il black friday @120540 mette in promo anche la voglia di vivere?

Cluster 1
Top terms: time, amazonhelp, 115821, prime, delivery, don, 115850, amazon, today, delivered
- @115850 I m happy because the 1 st time product of amazon recived in my smallest village today https://t.co/FfDj0Ivm4f
- @115830 really?? Is this huge box appropriate to send me my teeny weeny earring backs? An envelope will do next time! https://

PHASE 5 — Final Intent Taxonomy

In [8]:
# PHASE 5 — FINAL DATA-DERIVED INTENT TAXONOMY

import pandas as pd

taxonomy = pd.DataFrame([
    {
        "intent_id": "I01",
        "intent": "order_delivery",
        "description": "Order status, delayed delivery, estimated delivery date, delivery timing"
    },
    {
        "intent_id": "I02",
        "intent": "missing_package",
        "description": "Package marked delivered but missing, package not received, delivery disappearance"
    },
    {
        "intent_id": "I03",
        "intent": "return",
        "description": "Returning an item, return eligibility, return process or return instructions"
    },
    {
        "intent_id": "I04",
        "intent": "refund",
        "description": "Refund requested, refund status, refund amount or refund timing"
    },
    {
        "intent_id": "I05",
        "intent": "wrong_damaged_item",
        "description": "Wrong, damaged, defective, broken or unusable item"
    },
    {
        "intent_id": "I06",
        "intent": "payment_billing",
        "description": "Payment failure, unexpected charge, billing issue, payment method or card"
    },
    {
        "intent_id": "I07",
        "intent": "account",
        "description": "Account access, login, password, account settings or account-related issues"
    },
    {
        "intent_id": "I08",
        "intent": "prime_subscription",
        "description": "Prime membership, Prime benefits, subscription or Prime-related charges"
    },
    {
        "intent_id": "I09",
        "intent": "cancellation_modification",
        "description": "Order cancellation, changing an order, address or other order modifications"
    },
    {
        "intent_id": "I10",
        "intent": "general_information",
        "description": "General product, service, feature or policy information"
    },
    {
        "intent_id": "I11",
        "intent": "other_unclear",
        "description": "Unclear, miscellaneous or unsupported customer request"
    }
])

taxonomy.to_csv(
    "/content/amazonhelp_final_intent_taxonomy.csv",
    index=False
)

print("Number of intents:", len(taxonomy))

display(taxonomy)

Number of intents: 11


,intent_id,intent,description
0,I01,order_delivery,"Order status, delayed delivery, estimated deli..."
1,I02,missing_package,"Package marked delivered but missing, package ..."
2,I03,return,"Returning an item, return eligibility, return ..."
3,I04,refund,"Refund requested, refund status, refund amount..."
4,I05,wrong_damaged_item,"Wrong, damaged, defective, broken or unusable ..."
5,I06,payment_billing,"Payment failure, unexpected charge, billing is..."
6,I07,account,"Account access, login, password, account setti..."
7,I08,prime_subscription,"Prime membership, Prime benefits, subscription..."
8,I09,cancellation_modification,"Order cancellation, changing an order, address..."
9,I10,general_information,"General product, service, feature or policy in..."


PHASE 6 — Golden Set Creation & Human Labeling

In [9]:
# PHASE 6A — CREATE 200-EXAMPLE GOLDEN SET

import pandas as pd
import numpy as np

pairs_df = pd.read_csv(
    "/content/amazonhelp_conversation_pairs.csv"
)

taxonomy = pd.read_csv(
    "/content/amazonhelp_final_intent_taxonomy.csv"
)

# Reproducible random sample
golden_size = min(200, len(pairs_df))

golden = pairs_df.sample(
    n=golden_size,
    random_state=2026
).reset_index(drop=True)

# Add unique evaluation IDs
golden.insert(
    0,
    "example_id",
    [f"G{i:03d}" for i in range(1, golden_size + 1)]
)

# Human annotation columns
golden["intent"] = ""
golden["should_escalate"] = ""
golden["escalation_reason"] = ""

# Keep only fields needed for annotation/evaluation
golden = golden[
    [
        "example_id",
        "customer_tweet_id",
        "customer_text",
        "support_text",
        "customer_time",
        "support_time",
        "intent",
        "should_escalate",
        "escalation_reason"
    ]
]

golden.to_csv(
    "/content/amazonhelp_golden_set.csv",
    index=False
)

print("Golden set size:", len(golden))
print("Intent labels completed:", golden["intent"].ne("").sum())
print("Escalation labels completed:", golden["should_escalate"].ne("").sum())

display(golden.head(10))

Golden set size: 200
Intent labels completed: 0
Escalation labels completed: 0


,example_id,customer_tweet_id,customer_text,support_text,customer_time,support_time,intent,should_escalate,escalation_reason
0,G001,2262981,@AmazonHelp Filled out and sent.,@658813 Thanks for the update! A member of our...,Tue Nov 21 01:31:25 +0000 2017,Tue Nov 21 01:34:30 +0000 2017,,,
1,G002,2311986,@AmazonHelp https://t.co/pIZxMHZE22 https://t....,"@670494 Thanks for the screen grab, is that fr...",Mon Nov 13 15:37:17 +0000 2017,Mon Nov 13 16:00:00 +0000 2017,,,
2,G003,637228,@117795 Hi having real problems logging in it ...,@157530 I'm so sorry for your issues logging i...,Wed Nov 22 19:31:49 +0000 2017,Wed Nov 22 19:49:00 +0000 2017,,,
3,G004,2298426,"@AmazonHelp Yes, I’ve just been in contact wit...",@156774 We'd be happy to check on these with y...,Sun Nov 12 14:57:24 +0000 2017,Sun Nov 12 15:11:00 +0000 2017,,,
4,G005,859612,@AmazonHelp Order# 405-4831779-1488314\nThis s...,"@324100 Please don't share your order details,...",Fri Oct 20 18:00:34 +0000 2017,Fri Oct 20 18:28:45 +0000 2017,,,
5,G006,2910378,@116875 en mi cuenta me esta pidiendo cobro de...,@805705 Hola cinuxxx. Lamento saber del inconv...,Tue Nov 28 21:26:20 +0000 2017,Tue Nov 28 21:33:27 +0000 2017,,,
6,G007,355530,@117795 suggested resolution was take it back ...,@200470 Apologies - What return options were p...,Sun Oct 08 14:26:37 +0000 2017,Sun Oct 08 14:43:04 +0000 2017,,,
7,G008,2461317,@AmazonHelp Ya llego :) https://t.co/Ivpk9C3duM,"@704719 ¡Hola, Jorge! Estamos muy felices de q...",Thu Nov 16 21:26:56 +0000 2017,Thu Nov 16 21:39:30 +0000 2017,,,
8,G009,2810267,Non è che per il black friday @120540 mette in...,@495673 🤔 Giornata storta? 😅 ^FS,Wed Nov 22 06:45:13 +0000 2017,Wed Nov 22 06:52:29 +0000 2017,,,
9,G010,134998,@AmazonHelp I do not need it anymore now as it...,"@146420 Hi, I'm sorry there has been a delay. ...",Fri Nov 24 09:31:12 +0000 2017,Fri Nov 24 09:50:31 +0000 2017,,,


In [10]:
# PHASE 6B — HUMAN GOLDEN-SET LABELING

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

path = "/content/amazonhelp_golden_set.csv"

golden = pd.read_csv(path)

# Ensure annotation columns exist
for col in [
    "intent",
    "should_escalate",
    "escalation_reason"
]:
    if col not in golden.columns:
        golden[col] = ""

# Only work through the first 20 currently-unlabelled examples
unlabelled = golden[
    golden["intent"].fillna("").astype(str).str.strip() == ""
].index.tolist()

batch_indices = unlabelled[:20]

if len(batch_indices) == 0:
    print("All 200 examples are already labelled.")
else:

    position = 0

    intent_options = [
        "I01_order_delivery",
        "I02_missing_package",
        "I03_return",
        "I04_refund",
        "I05_wrong_damaged_item",
        "I06_payment_billing",
        "I07_account",
        "I08_prime_subscription",
        "I09_cancellation_modification",
        "I10_general_information",
        "I11_other_unclear"
    ]

    reason_options = [
        "None — safe to handle",
        "Explicit human request",
        "Sensitive account/payment issue",
        "Low-confidence or unclear request",
        "No safe historical resolution",
        "Complex issue requiring human intervention"
    ]

    customer_widget = widgets.HTML()

    intent_widget = widgets.Dropdown(
        options=intent_options,
        description="Intent:",
        layout=widgets.Layout(width="650px")
    )

    escalation_widget = widgets.Dropdown(
        options=["NO", "YES"],
        value="NO",
        description="Escalate:",
        layout=widgets.Layout(width="650px")
    )

    reason_widget = widgets.Dropdown(
        options=reason_options,
        value="None — safe to handle",
        description="Reason:",
        layout=widgets.Layout(width="650px")
    )

    save_button = widgets.Button(
        description="Save & Next",
        button_style="success"
    )

    output = widgets.Output()

    def load_example():

        global position

        with output:
            clear_output()

        if position >= len(batch_indices):

            golden.to_csv(
                path,
                index=False
            )

            with output:
                print("Batch complete.")
                print(
                    "Examples labelled in this batch:",
                    len(batch_indices)
                )
                print(
                    "Total examples labelled:",
                    (
                        golden["intent"]
                        .fillna("")
                        .astype(str)
                        .str.strip()
                        .ne("")
                        .sum()
                    )
                )

            save_button.disabled = True
            return

        idx = batch_indices[position]
        row = golden.loc[idx]

        customer_text = str(
            row["customer_text"]
        )

        support_text = str(
            row["support_text"]
        )

        customer_widget.value = f"""
        <h4>Example {position + 1}/20</h4>

        <p>
        <b>Customer message:</b><br>
        {customer_text}
        </p>

        <p>
        <b>Historical AmazonHelp response:</b><br>
        {support_text}
        </p>
        """

        # Reset widgets
        intent_widget.value = "I11_other_unclear"
        escalation_widget.value = "NO"
        reason_widget.value = "None — safe to handle"

    def save_next(_):

        global position

        idx = batch_indices[position]

        golden.loc[idx, "intent"] = (
            intent_widget.value
        )

        golden.loc[idx, "should_escalate"] = (
            escalation_widget.value
        )

        golden.loc[idx, "escalation_reason"] = (
            reason_widget.value
        )

        # Save immediately
        golden.to_csv(
            path,
            index=False
        )

        position += 1

        load_example()

    save_button.on_click(save_next)

    display(customer_widget)

    display(
        widgets.HTML(
            "<b>Choose the intent that best describes the "
            "CUSTOMER'S main problem.</b>"
        )
    )

    display(intent_widget)

    display(
        widgets.HTML(
            "<b>Escalation:</b> choose YES only when "
            "a human agent should handle the case."
        )
    )

    display(escalation_widget)

    display(reason_widget)

    display(save_button)

    display(output)

    load_example()

HTML(value='')

HTML(value="<b>Choose the intent that best describes the CUSTOMER'S main problem.</b>")

Dropdown(description='Intent:', layout=Layout(width='650px'), options=('I01_order_delivery', 'I02_missing_pack…

HTML(value='<b>Escalation:</b> choose YES only when a human agent should handle the case.')

Dropdown(description='Escalate:', layout=Layout(width='650px'), options=('NO', 'YES'), value='NO')

Dropdown(description='Reason:', layout=Layout(width='650px'), options=('None — safe to handle', 'Explicit huma…

Button(button_style='success', description='Save & Next', style=ButtonStyle())

Output()

/tmp/ipykernel_876/337123002.py:156: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'I11_other_unclear' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  golden.loc[idx, "intent"] = (
/tmp/ipykernel_876/337123002.py:160: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NO' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  golden.loc[idx, "should_escalate"] = (
/tmp/ipykernel_876/337123002.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Low-confidence or unclear request' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  golden.loc[idx, "escalation_reason"] = (


In [11]:
# PHASE 6B — FIX GOLDEN SET LABEL COLUMNS

import pandas as pd

path = "/content/amazonhelp_golden_set.csv"

golden = pd.read_csv(path)

# Explicitly make annotation columns strings
for col in [
    "intent",
    "should_escalate",
    "escalation_reason"
]:
    golden[col] = golden[col].fillna("").astype(str)

# Save corrected file
golden.to_csv(path, index=False)

print("Golden set fixed.")
print("Total examples:", len(golden))
print("Already labelled:", (golden["intent"].str.strip() != "").sum())
print("Remaining:", (golden["intent"].str.strip() == "").sum())

Golden set fixed.
Total examples: 200
Already labelled: 20
Remaining: 180


In [12]:
# PHASE 6C — HUMAN LABELING: EXAMPLES 21–40

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

path = "/content/amazonhelp_golden_set.csv"

golden = pd.read_csv(path)

# Force annotation columns to pandas string type
for col in [
    "intent",
    "should_escalate",
    "escalation_reason"
]:
    golden[col] = golden[col].fillna("").astype("string")

# Find unlabelled examples
unlabelled = golden.index[
    golden["intent"].fillna("").str.strip() == ""
].tolist()

# Next batch = maximum 20 examples
batch_indices = unlabelled[:20]

print("Examples in this batch:", len(batch_indices))

position = 0

intent_options = [
    "I01_order_delivery",
    "I02_missing_package",
    "I03_return",
    "I04_refund",
    "I05_wrong_damaged_item",
    "I06_payment_billing",
    "I07_account",
    "I08_prime_subscription",
    "I09_cancellation_modification",
    "I10_general_information",
    "I11_other_unclear"
]

reason_options = [
    "None — safe to handle",
    "Explicit human request",
    "Sensitive account/payment issue",
    "Low-confidence or unclear request",
    "No safe historical resolution",
    "Complex issue requiring human intervention"
]

customer_widget = widgets.HTML()

intent_widget = widgets.Dropdown(
    options=intent_options,
    value="I11_other_unclear",
    description="Intent:",
    layout=widgets.Layout(width="750px")
)

escalation_widget = widgets.Dropdown(
    options=["NO", "YES"],
    value="NO",
    description="Escalate:",
    layout=widgets.Layout(width="750px")
)

reason_widget = widgets.Dropdown(
    options=reason_options,
    value="None — safe to handle",
    description="Reason:",
    layout=widgets.Layout(width="750px")
)

save_button = widgets.Button(
    description="Save & Next",
    button_style="success"
)

output = widgets.Output()

def load_example():

    global position

    if position >= len(batch_indices):

        golden.to_csv(
            path,
            index=False
        )

        with output:
            clear_output()
            print("Batch complete.")
            print(
                "Total examples labelled:",
                golden["intent"].fillna("").str.strip().ne("").sum()
            )
            print(
                "Remaining:",
                golden["intent"].fillna("").str.strip().eq("").sum()
            )

        save_button.disabled = True
        return

    idx = batch_indices[position]
    row = golden.loc[idx]

    customer_text = str(row["customer_text"])
    support_text = str(row["support_text"])

    customer_widget.value = f"""
    <h4>Example {position + 1}/{len(batch_indices)}</h4>

    <p>
    <b>Customer message:</b><br>
    {customer_text}
    </p>

    <p>
    <b>Historical AmazonHelp response:</b><br>
    {support_text}
    </p>
    """

    intent_widget.value = "I11_other_unclear"
    escalation_widget.value = "NO"
    reason_widget.value = "None — safe to handle"


def save_next(_):

    global position

    idx = batch_indices[position]

    # Explicit string assignment
    golden.at[idx, "intent"] = str(
        intent_widget.value
    )

    golden.at[idx, "should_escalate"] = str(
        escalation_widget.value
    )

    golden.at[idx, "escalation_reason"] = str(
        reason_widget.value
    )

    # Save immediately
    golden.to_csv(
        path,
        index=False
    )

    position += 1

    load_example()


save_button.on_click(save_next)

display(customer_widget)
display(intent_widget)
display(escalation_widget)
display(reason_widget)
display(save_button)
display(output)

load_example()

Examples in this batch: 20


HTML(value='')

Dropdown(description='Intent:', index=10, layout=Layout(width='750px'), options=('I01_order_delivery', 'I02_mi…

Dropdown(description='Escalate:', layout=Layout(width='750px'), options=('NO', 'YES'), value='NO')

Dropdown(description='Reason:', layout=Layout(width='750px'), options=('None — safe to handle', 'Explicit huma…

Button(button_style='success', description='Save & Next', style=ButtonStyle())

Output()

In [13]:
import pandas as pd

path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(path)

for col in ["intent", "should_escalate", "escalation_reason"]:
    golden[col] = golden[col].fillna("").astype("string")

remaining = golden[
    golden["intent"].fillna("").astype(str).str.strip() == ""
]

print(f"Labelled: {len(golden) - len(remaining)} / {len(golden)}")
print(f"Remaining: {len(remaining)}")

Labelled: 40 / 200
Remaining: 160


In [14]:
import pandas as pd
from IPython.display import display, Markdown
import ipywidgets as widgets

path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(path)

# Keep annotation columns as strings
for col in ["intent", "should_escalate", "escalation_reason"]:
    golden[col] = golden[col].fillna("").astype("string")

# Find all unlabelled examples
remaining_idx = golden[
    golden["intent"].fillna("").astype(str).str.strip() == ""
].index.tolist()

# Take next 20
batch_idx = remaining_idx[:20]

intents = [
    "I01_order_delivery",
    "I02_missing_package",
    "I03_return",
    "I04_refund",
    "I05_wrong_damaged_item",
    "I06_payment_billing",
    "I07_account",
    "I08_prime_subscription",
    "I09_cancellation_modification",
    "I10_general_information",
    "I11_other_unclear"
]

reasons = [
    "None — safe to handle",
    "Low-confidence or unclear request",
    "Explicit human request",
    "Sensitive account/payment issue",
    "Complex issue requiring human intervention"
]

display(Markdown(
    f"### Examples in this batch: {len(batch_idx)}  \n"
    f"**Current progress: {len(golden) - len(remaining_idx)} / {len(golden)}**  \n"
    f"**After this batch: {len(golden) - len(remaining_idx) + len(batch_idx)} / {len(golden)}**"
))

for number, idx in enumerate(batch_idx, start=1):

    row = golden.loc[idx]

    display(Markdown(
        f"---\n"
        f"### Example {number}/{len(batch_idx)}\n\n"
        f"**Customer message:**  \n"
        f"{row['customer_text']}\n\n"
        f"**Historical AmazonHelp response:**  \n"
        f"{row['support_text']}"
    ))

    intent_dd = widgets.Dropdown(
        options=intents,
        value=None,
        description="Intent:",
        layout=widgets.Layout(width="650px")
    )

    escalate_dd = widgets.Dropdown(
        options=["NO", "YES"],
        value=None,
        description="Escalate:",
        layout=widgets.Layout(width="650px")
    )

    reason_dd = widgets.Dropdown(
        options=reasons,
        value=None,
        description="Reason:",
        layout=widgets.Layout(width="650px")
    )

    save_btn = widgets.Button(
        description="Save & Next",
        button_style="success",
        layout=widgets.Layout(width="150px")
    )

    output = widgets.Output()

    def save_label(
        b,
        idx=idx,
        intent_dd=intent_dd,
        escalate_dd=escalate_dd,
        reason_dd=reason_dd,
        save_btn=save_btn,
        output=output
    ):
        with output:
            if intent_dd.value is None or escalate_dd.value is None or reason_dd.value is None:
                print("Please select all three fields.")
                return

            golden.at[idx, "intent"] = intent_dd.value
            golden.at[idx, "should_escalate"] = escalate_dd.value
            golden.at[idx, "escalation_reason"] = reason_dd.value

            golden.to_csv(path, index=False)

            save_btn.disabled = True
            print("Saved.")

    save_btn.on_click(save_label)

    display(intent_dd)
    display(escalate_dd)
    display(reason_dd)
    display(save_btn)
    display(output)

print("Batch ready. Label each example and click Save & Next.")

### Examples in this batch: 20  
**Current progress: 40 / 200**  
**After this batch: 60 / 200**

---
### Example 1/20

**Customer message:**  
@AmazonHelp @152011 DEAR AMAZONE FIND THE ORDER NO.-403-79045985510732 WHAT IS ISSUE ABOUT PARCEL RETURN TO UR END AND CONTAIN MOBILE IS REC AT UR LOACTION . IN YOUR WAREHOUSE TOTALLY THIEF AND LYING STAFF.WHERE IS MY MOBILE???HOW MANY DAYS MY MOBILE GET IT BACK ON ARGENT BASIS.

**Historical AmazonHelp response:**  
@152012 We apologize for the inconvenience regarding the order. We'd like to take a closer look into this issue. Kindly drop your details here: https://t.co/beaaDm0muc and we’ll contact you soon. 1/2 ^BV

Dropdown(description='Intent:', layout=Layout(width='650px'), options=('I01_order_delivery', 'I02_missing_pack…

Dropdown(description='Escalate:', layout=Layout(width='650px'), options=('NO', 'YES'), value=None)

Dropdown(description='Reason:', layout=Layout(width='650px'), options=('None — safe to handle', 'Low-confidenc…

...

Dropdown(description='Escalate:', layout=Layout(width='650px'), options=('NO', 'YES'), value=None)

Dropdown(description='Reason:', layout=Layout(width='650px'), options=('None — safe to handle', 'Low-confidenc…

Button(button_style='success', description='Save & Next', layout=Layout(width='150px'), style=ButtonStyle())

Output()

Batch ready. Label each example and click Save & Next.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Please select all three fields.
Saved.
Please select all three fields.
Please select all three fields.
Saved.
Saved.
Saved.
Saved.
Saved.
Saved.
Saved.
Saved.
Saved.


In [16]:
import pandas as pd

path = "/content/amazonhelp_golden_set.csv"
g = pd.read_csv(path)

g["intent"] = g["intent"].fillna("").astype(str).str.strip()
g["should_escalate"] = g["should_escalate"].fillna("").astype(str).str.strip()
g["escalation_reason"] = g["escalation_reason"].fillna("").astype(str).str.strip()

complete = (
    (g["intent"] != "") &
    (g["should_escalate"] != "") &
    (g["escalation_reason"] != "")
)

print("TOTAL:", len(g))
print("COMPLETED:", complete.sum())
print("REMAINING:", (~complete).sum())

print("\nCompleted examples:")
print(g.loc[complete, ["example_id", "intent", "should_escalate", "escalation_reason"]].tail(10).to_string(index=False))

TOTAL: 200
COMPLETED: 59
REMAINING: 141

Completed examples:
example_id                  intent should_escalate                          escalation_reason
      G051      I01_order_delivery             YES Complex issue requiring human intervention
      G052      I01_order_delivery              NO                      None — safe to handle
      G053     I06_payment_billing             YES            Sensitive account/payment issue
      G054       I11_other_unclear             YES Complex issue requiring human intervention
      G055       I11_other_unclear              NO          Low-confidence or unclear request
      G056 I10_general_information             YES Complex issue requiring human intervention
      G057      I01_order_delivery              NO                      None — safe to handle
      G058  I05_wrong_damaged_item              NO                      None — safe to handle
      G059             I07_account             YES            Sensitive account/payment issue

In [17]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown

path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(path)

for col in ["intent", "should_escalate", "escalation_reason"]:
    golden[col] = golden[col].fillna("").astype("string")

# Find all remaining unlabelled examples
remaining = golden[
    golden["intent"].fillna("").astype(str).str.strip() == ""
]

labelled = len(golden) - len(remaining)

print(f"Labelled: {labelled} / {len(golden)}")
print(f"Remaining: {len(remaining)}")

intents = [
    "I01_order_delivery",
    "I02_missing_package",
    "I03_return",
    "I04_refund",
    "I05_wrong_damaged_item",
    "I06_payment_billing",
    "I07_account",
    "I08_prime_subscription",
    "I09_cancellation_modification",
    "I10_general_information",
    "I11_other_unclear"
]

reasons = [
    "None — safe to handle",
    "Low-confidence or unclear request",
    "Explicit human request",
    "Sensitive account/payment issue",
    "Complex issue requiring human intervention"
]

for number, idx in enumerate(remaining.index, start=labelled + 1):

    row = golden.loc[idx]

    display(Markdown(
        f"## Example {number}/200\n\n"
        f"**Customer message:**\n\n"
        f"{row['customer_text']}\n\n"
        f"**Historical AmazonHelp response:**\n\n"
        f"{row['support_text']}"
    ))

    intent = widgets.Dropdown(
        options=[""] + intents,
        value="",
        description="Intent:",
        layout=widgets.Layout(width="750px")
    )

    escalate = widgets.Dropdown(
        options=["", "NO", "YES"],
        value="",
        description="Escalate:",
        layout=widgets.Layout(width="750px")
    )

    reason = widgets.Dropdown(
        options=[""] + reasons,
        value="",
        description="Reason:",
        layout=widgets.Layout(width="750px")
    )

    save_button = widgets.Button(
        description="Save & Next",
        button_style="success",
        layout=widgets.Layout(width="150px")
    )

    output = widgets.Output()

    def save_label(
        b,
        idx=idx,
        intent=intent,
        escalate=escalate,
        reason=reason,
        save_button=save_button,
        output=output
    ):
        with output:

            if not intent.value or not escalate.value or not reason.value:
                print("Please select all 3 fields.")
                return

            golden.at[idx, "intent"] = intent.value
            golden.at[idx, "should_escalate"] = escalate.value
            golden.at[idx, "escalation_reason"] = reason.value

            golden.to_csv(path, index=False)

            save_button.disabled = True
            print("Saved successfully.")

    save_button.on_click(save_label)

    display(intent)
    display(escalate)
    display(reason)
    display(save_button)
    display(output)

Labelled: 59 / 200
Remaining: 141


## Example 60/200

**Customer message:**

@AmazonHelp And I'm on the phone with you guys right now. :)

**Historical AmazonHelp response:**

@135279 Thanks for letting us know. Please keep us updated with the outcome. ^LI

Dropdown(description='Intent:', layout=Layout(width='750px'), options=('', 'I01_order_delivery', 'I02_missing_…

Dropdown(description='Escalate:', layout=Layout(width='750px'), options=('', 'NO', 'YES'), value='')

Dropdown(description='Reason:', layout=Layout(width='750px'), options=('', 'None — safe to handle', 'Low-confi…

...

Dropdown(description='Intent:', layout=Layout(width='750px'), options=('', 'I01_order_delivery', 'I02_missing_…

Dropdown(description='Escalate:', layout=Layout(width='750px'), options=('', 'NO', 'YES'), value='')

Dropdown(description='Reason:', layout=Layout(width='750px'), options=('', 'None — safe to handle', 'Low-confi…

Button(button_style='success', description='Save & Next', layout=Layout(width='150px'), style=ButtonStyle())

Output()

In [18]:
import pandas as pd

path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(path)

for col in ["intent", "should_escalate", "escalation_reason"]:
    golden[col] = golden[col].fillna("").astype(str)

labelled = golden[
    golden["intent"].str.strip() != ""
]

print("ACTUAL SAVED PROGRESS")
print(f"Labelled: {len(labelled)} / {len(golden)}")
print(f"Remaining: {len(golden) - len(labelled)}")

if len(labelled) > 0:
    print(f"Last labelled row: {labelled.index[-1]}")

ACTUAL SAVED PROGRESS
Labelled: 200 / 200
Remaining: 0
Last labelled row: 199


Phase 7A — Golden Set Validation

In [19]:
import pandas as pd

golden_path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(golden_path)

valid_intents = {
    "I01_order_delivery",
    "I02_missing_package",
    "I03_return",
    "I04_refund",
    "I05_wrong_damaged_item",
    "I06_payment_billing",
    "I07_account",
    "I08_prime_subscription",
    "I09_cancellation_modification",
    "I10_general_information",
    "I11_other_unclear"
}

valid_reasons = {
    "None — safe to handle",
    "Low-confidence or unclear request",
    "Explicit human request",
    "Sensitive account/payment issue",
    "Complex issue requiring human intervention"
}

print("Total examples:", len(golden))
print("Missing intents:", golden["intent"].isna().sum())
print("Missing escalation labels:", golden["should_escalate"].isna().sum())
print("Missing reasons:", golden["escalation_reason"].isna().sum())
print("Duplicate example IDs:", golden["example_id"].duplicated().sum())

invalid_intents = set(golden["intent"].dropna()) - valid_intents
invalid_escalation = set(golden["should_escalate"].dropna()) - {"YES", "NO"}
invalid_reasons = set(golden["escalation_reason"].dropna()) - valid_reasons

print("\nInvalid intents:", invalid_intents)
print("Invalid escalation values:", invalid_escalation)
print("Invalid reasons:", invalid_reasons)

print("\nIntent distribution:")
print(golden["intent"].value_counts())

print("\nEscalation distribution:")
print(golden["should_escalate"].value_counts())

if (
    len(golden) == 200
    and golden["intent"].notna().all()
    and golden["should_escalate"].notna().all()
    and golden["escalation_reason"].notna().all()
    and golden["example_id"].duplicated().sum() == 0
    and not invalid_intents
    and not invalid_escalation
    and not invalid_reasons
):
    print("\nVALIDATION PASSED")
else:
    print("\nVALIDATION FAILED — fix the issues above")

Total examples: 200
Missing intents: 0
Missing escalation labels: 0
Missing reasons: 0
Duplicate example IDs: 0

Invalid intents: set()
Invalid escalation values: set()
Invalid reasons: set()

Intent distribution:
intent
I11_other_unclear                52
I01_order_delivery               52
I10_general_information          26
I05_wrong_damaged_item           13
I07_account                      12
I02_missing_package              10
I08_prime_subscription           10
I06_payment_billing               9
I03_return                        6
I09_cancellation_modification     5
I04_refund                        5
Name: count, dtype: int64

Escalation distribution:
should_escalate
NO     108
YES     92
Name: count, dtype: int64

VALIDATION PASSED


In [20]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

golden_path = "/content/amazonhelp_golden_set.csv"
golden = pd.read_csv(golden_path)

# Majority class
majority_class = golden["intent"].value_counts().idxmax()

# Predict majority class for every example
y_true = golden["intent"]
y_pred = [majority_class] * len(golden)

accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Majority Class:", majority_class)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Save baseline result
baseline_results = pd.DataFrame([{
    "baseline": "Majority Classifier",
    "majority_class": majority_class,
    "accuracy": accuracy,
    "precision_weighted": precision,
    "recall_weighted": recall,
    "f1_weighted": f1
}])

baseline_results.to_csv(
    "/content/majority_baseline_results.csv",
    index=False
)

print("\nSaved: /content/majority_baseline_results.csv")

Majority Class: I11_other_unclear
Accuracy:  0.2600
Precision: 0.0676
Recall:    0.2600
F1 Score:  0.1073

Saved: /content/majority_baseline_results.csv


Phase 7C — Simple ML Baseline

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")

X = golden["customer_text"].fillna("")
y = golden["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=10000
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("TF-IDF + Logistic Regression")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

baseline_ml = pd.DataFrame([{
    "baseline": "TF-IDF + Logistic Regression",
    "accuracy": accuracy,
    "precision_weighted": precision,
    "recall_weighted": recall,
    "f1_weighted": f1
}])

baseline_ml.to_csv(
    "/content/tfidf_logistic_baseline_results.csv",
    index=False
)

print("\nSaved: /content/tfidf_logistic_baseline_results.csv")

TF-IDF + Logistic Regression
Accuracy:  0.3200
Precision: 0.2738
Recall:    0.3200
F1 Score:  0.2942

Saved: /content/tfidf_logistic_baseline_results.csv


Phase 7D — Build the Retrieval-Based Support Agent

In [22]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load historical AmazonHelp conversation pairs
pairs = pd.read_csv("/content/amazonhelp_conversation_pairs.csv")

pairs["customer_text"] = pairs["customer_text"].fillna("").astype(str)
pairs["support_text"] = pairs["support_text"].fillna("").astype(str)

# TF-IDF index over historical customer messages
retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)

retrieval_matrix = retrieval_vectorizer.fit_transform(
    pairs["customer_text"]
)

print("Historical conversations:", len(pairs))
print("TF-IDF matrix shape:", retrieval_matrix.shape)

def retrieve_cases(customer_message, top_k=3):
    query_vector = retrieval_vectorizer.transform([customer_message])
    scores = cosine_similarity(query_vector, retrieval_matrix).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = pairs.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]

    return results[
        ["customer_text", "support_text", "similarity"]
    ].reset_index(drop=True)

# Test retrieval
test_message = "My package has not arrived yet and the delivery is delayed."

results = retrieve_cases(test_message, top_k=3)

print("\nTest customer message:")
print(test_message)

print("\nTop retrieved historical cases:")
for i, row in results.iterrows():
    print(f"\nCase {i+1} | Similarity: {row['similarity']:.4f}")
    print("Customer:", row["customer_text"][:250])
    print("Support:", row["support_text"][:250])

Historical conversations: 168814
TF-IDF matrix shape: (168814, 30000)

Test customer message:
My package has not arrived yet and the delivery is delayed.

Top retrieved historical cases:

Case 1 | Similarity: 0.5654
Customer: @AmazonHelp Yes, has not arrived yet
Support: @566298 Oh no! Have we missed the delivery date provided in your confirmation e-mail? Let us know- we're here to help! ^BL

Case 2 | Similarity: 0.4540
Customer: @AmazonHelp Hey so my package has not arrived yet. Can I get this sorted please
Support: @119403 While we can't access order or account details through social media, we'd still like to help! Please reach us directly via phone or chat so we can investigate: https://t.co/JzP7hlA23B ^CS

Case 3 | Similarity: 0.3882
Customer: Hey @115850 , my package has not arrived yet @AmazonHelp  but ur tracking shows tat it shld have been delivered by now
Support: @340085 Sorry for the delay, Aman. Please fill in your details here: https://t.co/FkTkJU6VVi and we will get back 

Phase 7E — Build the Support Agent

In [23]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load trained intent model and historical data
golden = pd.read_csv("/content/amazonhelp_golden_set.csv")
pairs = pd.read_csv("/content/amazonhelp_conversation_pairs.csv")

pairs["customer_text"] = pairs["customer_text"].fillna("").astype(str)
pairs["support_text"] = pairs["support_text"].fillna("").astype(str)

def support_agent(customer_message, top_k=3):
    # 1. Intent classification
    probabilities = model.predict_proba([customer_message])[0]
    classes = model.classes_

    best_idx = np.argmax(probabilities)
    intent = classes[best_idx]
    confidence = probabilities[best_idx]

    # 2. Retrieve historical cases
    retrieved = retrieve_cases(customer_message, top_k=top_k)
    best_similarity = retrieved.iloc[0]["similarity"]

    # 3. Escalation rules
    text = customer_message.lower()

    explicit_human = any(
        phrase in text
        for phrase in [
            "human", "agent", "representative",
            "speak to someone", "talk to someone",
            "contact me", "call me"
        ]
    )

    sensitive_issue = (
        intent in ["I06_payment_billing", "I07_account"]
        and any(
            word in text
            for word in [
                "hack", "hacked", "stolen", "fraud",
                "charge", "charged", "payment",
                "card", "bank", "password", "login"
            ]
        )
    )

    low_confidence = confidence < 0.35
    weak_retrieval = best_similarity < 0.20

    if explicit_human:
        decision = "ESCALATE"
        reason = "Explicit human request"
    elif sensitive_issue:
        decision = "ESCALATE"
        reason = "Sensitive account/payment issue"
    elif low_confidence:
        decision = "ESCALATE"
        reason = "Low-confidence or unclear request"
    elif weak_retrieval:
        decision = "ESCALATE"
        reason = "No safe historical resolution"
    else:
        decision = "AUTO-HANDLE"
        reason = "Historical resolution found"

    # 4. Grounded reply
    draft_reply = retrieved.iloc[0]["support_text"]

    return {
        "intent": intent,
        "confidence": round(float(confidence), 4),
        "retrieved_cases": retrieved,
        "retrieval_score": round(float(best_similarity), 4),
        "draft_reply": draft_reply,
        "decision": decision,
        "reason": reason
    }

# Test the complete agent
test_message = "My package has not arrived yet and the delivery is delayed."

result = support_agent(test_message)

print("Customer:", test_message)
print("\nIntent:", result["intent"])
print("Confidence:", result["confidence"])
print("Retrieval score:", result["retrieval_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGrounded draft reply:")
print(result["draft_reply"])

Customer: My package has not arrived yet and the delivery is delayed.

Intent: I01_order_delivery
Confidence: 0.1337
Retrieval score: 0.5654
Decision: ESCALATE
Reason: Low-confidence or unclear request

Grounded draft reply:
@566298 Oh no! Have we missed the delivery date provided in your confirmation e-mail? Let us know- we're here to help! ^BL


In [24]:
def support_agent(customer_message, top_k=3):
    # Intent prediction
    probabilities = model.predict_proba([customer_message])[0]
    classes = model.classes_

    best_idx = np.argmax(probabilities)
    intent = classes[best_idx]
    classifier_confidence = float(probabilities[best_idx])

    # Historical retrieval
    retrieved = retrieve_cases(customer_message, top_k=top_k)
    retrieval_score = float(retrieved.iloc[0]["similarity"])

    text = customer_message.lower()

    # Explicit request for a human
    explicit_human = any(
        phrase in text
        for phrase in [
            "human",
            "agent",
            "representative",
            "speak to someone",
            "talk to someone",
            "contact me",
            "call me"
        ]
    )

    # Sensitive account/payment issue
    sensitive_issue = (
        intent in ["I06_payment_billing", "I07_account"]
        and any(
            word in text
            for word in [
                "hack", "hacked", "stolen", "fraud",
                "charge", "charged", "payment",
                "card", "bank", "password", "login"
            ]
        )
    )

    # Decision policy
    if explicit_human:
        decision = "ESCALATE"
        reason = "Explicit human request"

    elif sensitive_issue:
        decision = "ESCALATE"
        reason = "Sensitive account/payment issue"

    elif retrieval_score < 0.20:
        decision = "ESCALATE"
        reason = "No safe historical resolution"

    else:
        decision = "AUTO-HANDLE"
        reason = "Relevant historical resolution found"

    # Grounded response
    draft_reply = retrieved.iloc[0]["support_text"]

    return {
        "intent": intent,
        "confidence": round(classifier_confidence, 4),
        "retrieved_cases": retrieved,
        "retrieval_score": round(retrieval_score, 4),
        "draft_reply": draft_reply,
        "decision": decision,
        "reason": reason
    }


# Test
test_message = "My package has not arrived yet and the delivery is delayed."

result = support_agent(test_message)

print("Customer:", test_message)
print("\nIntent:", result["intent"])
print("Classifier confidence:", result["confidence"])
print("Retrieval score:", result["retrieval_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGrounded draft reply:")
print(result["draft_reply"])

Customer: My package has not arrived yet and the delivery is delayed.

Intent: I01_order_delivery
Classifier confidence: 0.1337
Retrieval score: 0.5654
Decision: AUTO-HANDLE
Reason: Relevant historical resolution found

Grounded draft reply:
@566298 Oh no! Have we missed the delivery date provided in your confirmation e-mail? Let us know- we're here to help! ^BL


Phase 7F — Evaluate the Support Agent

In [25]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")

results = []

for _, row in golden.iterrows():
    result = support_agent(row["customer_text"])

    results.append({
        "example_id": row["example_id"],
        "customer_text": row["customer_text"],
        "true_intent": row["intent"],
        "predicted_intent": result["intent"],
        "classifier_confidence": result["confidence"],
        "retrieval_score": result["retrieval_score"],
        "true_escalation": row["should_escalate"],
        "predicted_decision": result["decision"],
        "escalation_reason": result["reason"],
        "draft_reply": result["draft_reply"]
    })

agent_results = pd.DataFrame(results)

# Intent metrics
intent_accuracy = accuracy_score(
    agent_results["true_intent"],
    agent_results["predicted_intent"]
)

intent_precision, intent_recall, intent_f1, _ = precision_recall_fscore_support(
    agent_results["true_intent"],
    agent_results["predicted_intent"],
    average="weighted",
    zero_division=0
)

# Convert escalation labels to YES/NO
predicted_escalation = agent_results["predicted_decision"].map({
    "ESCALATE": "YES",
    "AUTO-HANDLE": "NO"
})

escalation_accuracy = accuracy_score(
    agent_results["true_escalation"],
    predicted_escalation
)

esc_precision, esc_recall, esc_f1, _ = precision_recall_fscore_support(
    agent_results["true_escalation"],
    predicted_escalation,
    pos_label="YES",
    average="binary",
    zero_division=0
)

# Retrieval success
retrieval_success = (
    agent_results["retrieval_score"] >= 0.20
).mean()

# Auto-handle rate
auto_handle_rate = (
    agent_results["predicted_decision"] == "AUTO-HANDLE"
).mean()

print("SUPPORT AGENT EVALUATION")
print("=" * 40)

print(f"Examples evaluated:       {len(agent_results)}")

print("\nINTENT CLASSIFICATION")
print(f"Accuracy:                 {intent_accuracy:.4f}")
print(f"Weighted Precision:       {intent_precision:.4f}")
print(f"Weighted Recall:          {intent_recall:.4f}")
print(f"Weighted F1:              {intent_f1:.4f}")

print("\nESCALATION")
print(f"Accuracy:                 {escalation_accuracy:.4f}")
print(f"Precision:                {esc_precision:.4f}")
print(f"Recall:                   {esc_recall:.4f}")
print(f"F1:                       {esc_f1:.4f}")

print("\nRETRIEVAL")
print(f"Retrieval Success:        {retrieval_success:.4f}")

print("\nAUTOMATION")
print(f"Auto-handle Rate:         {auto_handle_rate:.4f}")

# Save complete evaluation
agent_results.to_csv(
    "/content/golden_agent_results.csv",
    index=False
)

metrics = {
    "examples": len(agent_results),
    "intent_accuracy": intent_accuracy,
    "intent_precision_weighted": intent_precision,
    "intent_recall_weighted": intent_recall,
    "intent_f1_weighted": intent_f1,
    "escalation_accuracy": escalation_accuracy,
    "escalation_precision": esc_precision,
    "escalation_recall": esc_recall,
    "escalation_f1": esc_f1,
    "retrieval_success": retrieval_success,
    "auto_handle_rate": auto_handle_rate
}

pd.DataFrame([metrics]).to_json(
    "/content/agent_evaluation_metrics.json",
    orient="records",
    indent=2
)

print("\nSaved:")
print("/content/golden_agent_results.csv")
print("/content/agent_evaluation_metrics.json")

SUPPORT AGENT EVALUATION
Examples evaluated:       200

INTENT CLASSIFICATION
Accuracy:                 0.8300
Weighted Precision:       0.8363
Weighted Recall:          0.8300
Weighted F1:              0.8304

ESCALATION
Accuracy:                 0.5700
Precision:                0.7500
Recall:                   0.0978
F1:                       0.1731

RETRIEVAL
Retrieval Success:        0.9900

AUTOMATION
Auto-handle Rate:         0.9400

Saved:
/content/golden_agent_results.csv
/content/agent_evaluation_metrics.json


In [27]:
import pandas as pd

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")
pairs = pd.read_csv("/content/amazonhelp_conversation_pairs.csv")

golden_ids = set(golden["customer_tweet_id"].astype(str))
pair_ids = set(pairs["customer_tweet_id"].astype(str))

overlap_ids = golden_ids & pair_ids

print("Golden examples:", len(golden))
print("Unique golden IDs:", len(golden_ids))
print("Historical pairs:", len(pairs))
print("Unique overlapping golden IDs:", len(overlap_ids))
print("Actual retrieval leakage rate:", round(len(overlap_ids) / len(golden_ids) * 100, 2), "%")

duplicate_matches = (
    pairs[pairs["customer_tweet_id"].astype(str).isin(golden_ids)]
    ["customer_tweet_id"]
    .astype(str)
    .value_counts()
)

print("\nHistorical rows belonging to golden IDs:", len(pairs[pairs["customer_tweet_id"].astype(str).isin(golden_ids)]))
print("Golden IDs with duplicate historical rows:", (duplicate_matches > 1).sum())

Golden examples: 200
Unique golden IDs: 200
Historical pairs: 168814
Unique overlapping golden IDs: 200
Actual retrieval leakage rate: 100.0 %

Historical rows belonging to golden IDs: 238
Golden IDs with duplicate historical rows: 32


Phase 7G — Leakage-safe retrieval evaluation

In [28]:
import pandas as pd
import numpy as np

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")
pairs = pd.read_csv("/content/amazonhelp_conversation_pairs.csv")

retrieval_results = []

for _, row in golden.iterrows():
    customer_message = row["customer_text"]
    golden_id = str(row["customer_tweet_id"])

    query_vector = retrieval_vectorizer.transform([customer_message])
    scores = (retrieval_matrix @ query_vector.T).toarray().ravel()

    candidate_pairs = pairs.copy()
    candidate_pairs["similarity"] = scores

    # Remove the golden example itself and any duplicate rows
    candidate_pairs = candidate_pairs[
        candidate_pairs["customer_tweet_id"].astype(str) != golden_id
    ]

    top_case = candidate_pairs.nlargest(1, "similarity").iloc[0]

    retrieval_results.append({
        "example_id": row["example_id"],
        "customer_text": customer_message,
        "golden_customer_tweet_id": golden_id,
        "top_similarity": float(top_case["similarity"]),
        "retrieved_customer_text": top_case["customer_text"],
        "retrieved_support_text": top_case["support_text"]
    })

leakage_safe_retrieval = pd.DataFrame(retrieval_results)

retrieval_success = (
    leakage_safe_retrieval["top_similarity"] >= 0.20
).mean()

print("LEAKAGE-SAFE RETRIEVAL EVALUATION")
print("Examples evaluated:", len(leakage_safe_retrieval))
print(f"Retrieval Success: {retrieval_success:.4f}")
print(f"Mean Similarity: {leakage_safe_retrieval['top_similarity'].mean():.4f}")
print(f"Median Similarity: {leakage_safe_retrieval['top_similarity'].median():.4f}")
print(f"Minimum Similarity: {leakage_safe_retrieval['top_similarity'].min():.4f}")

leakage_safe_retrieval.to_csv(
    "/content/leakage_safe_retrieval_results.csv",
    index=False
)

print("\nSaved:")
print("/content/leakage_safe_retrieval_results.csv")

LEAKAGE-SAFE RETRIEVAL EVALUATION
Examples evaluated: 200
Retrieval Success: 0.9850
Mean Similarity: 0.4386
Median Similarity: 0.3715
Minimum Similarity: 0.0000

Saved:
/content/leakage_safe_retrieval_results.csv


In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")

X = golden["customer_text"].fillna("")
y = golden["intent"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = np.empty(len(golden), dtype=object)
oof_confidence = np.zeros(len(golden))

for train_idx, test_idx in skf.split(X, y):

    fold_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
                max_features=10000
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced"
            )
        )
    ])

    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])

    oof_predictions[test_idx] = fold_model.predict(X.iloc[test_idx])

    probabilities = fold_model.predict_proba(X.iloc[test_idx])
    oof_confidence[test_idx] = probabilities.max(axis=1)

intent_accuracy = accuracy_score(y, oof_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    y,
    oof_predictions,
    average="weighted",
    zero_division=0
)

print("5-FOLD OUT-OF-FOLD INTENT EVALUATION")
print("Examples evaluated:", len(golden))
print(f"Accuracy:                 {intent_accuracy:.4f}")
print(f"Weighted Precision:       {precision:.4f}")
print(f"Weighted Recall:          {recall:.4f}")
print(f"Weighted F1:              {f1:.4f}")
print(f"Mean Confidence:          {oof_confidence.mean():.4f}")

oof_results = golden[
    ["example_id", "customer_tweet_id", "customer_text", "intent"]
].copy()

oof_results["predicted_intent"] = oof_predictions
oof_results["classifier_confidence"] = oof_confidence

oof_results.to_csv(
    "/content/oof_intent_results.csv",
    index=False
)

print("\nSaved:")
print("/content/oof_intent_results.csv")

5-FOLD OUT-OF-FOLD INTENT EVALUATION
Examples evaluated: 200
Accuracy:                 0.4350
Weighted Precision:       0.3779
Weighted Recall:          0.4350
Weighted F1:              0.3936
Mean Confidence:          0.1272

Saved:
/content/oof_intent_results.csv


In [30]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(golden["intent"].unique())

cm = confusion_matrix(
    golden["intent"],
    oof_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("CONFUSION MATRIX")
print(cm_df)

print("\nPER-INTENT PERFORMANCE")

from sklearn.metrics import classification_report

print(
    classification_report(
        golden["intent"],
        oof_predictions,
        labels=labels,
        zero_division=0
    )
)

cm_df.to_csv("/content/intent_confusion_matrix.csv")

print("\nSaved:")
print("/content/intent_confusion_matrix.csv")

CONFUSION MATRIX
                               I01_order_delivery  I02_missing_package  \
I01_order_delivery                             34                    1   
I02_missing_package                             4                    3   
I03_return                                      2                    1   
I04_refund                                      1                    0   
I05_wrong_damaged_item                          5                    0   
I06_payment_billing                             3                    0   
I07_account                                     2                    0   
I08_prime_subscription                          3                    0   
I09_cancellation_modification                   4                    1   
I10_general_information                         7                    1   
I11_other_unclear                              10                    0   

                               I03_return  I04_refund  I05_wrong_damaged_item  \
I01_order_del

In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")

X = golden["customer_text"].fillna("")
y = golden["intent"]

models = {
    "Word TF-IDF + Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1,2),
            min_df=1,
            max_features=15000,
            sublinear_tf=True
        )),
        ("classifier", LogisticRegression(
            max_iter=3000,
            class_weight="balanced"
        ))
    ]),

    "Word + Character TF-IDF + Logistic Regression": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(
                lowercase=True,
                ngram_range=(1,2),
                min_df=1,
                max_features=15000,
                sublinear_tf=True
            )),
            ("char", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3,5),
                min_df=1,
                max_features=15000,
                sublinear_tf=True
            ))
        ])),
        ("classifier", LogisticRegression(
            max_iter=3000,
            class_weight="balanced"
        ))
    ]),

    "Word TF-IDF + Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1,2),
            min_df=1,
            max_features=15000,
            sublinear_tf=True
        )),
        ("classifier", LinearSVC(
            class_weight="balanced"
        ))
    ])
}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

comparison = []

for model_name, clf in models.items():

    predictions = np.empty(len(X), dtype=object)

    for train_idx, test_idx in skf.split(X, y):

        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions[test_idx] = clf.predict(X.iloc[test_idx])

    accuracy = accuracy_score(y, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y,
        predictions,
        average="weighted",
        zero_division=0
    )

    comparison.append({
        "model": model_name,
        "accuracy": accuracy,
        "weighted_precision": precision,
        "weighted_recall": recall,
        "weighted_f1": f1
    })

comparison_df = pd.DataFrame(comparison)

print("INTENT MODEL COMPARISON — 5-FOLD OOF")
print(comparison_df.to_string(index=False))

comparison_df.to_csv(
    "/content/intent_model_comparison.csv",
    index=False
)

best_model = comparison_df.loc[
    comparison_df["weighted_f1"].idxmax()
]

print("\nBEST MODEL:")
print(best_model.to_string())

print("\nSaved:")
print("/content/intent_model_comparison.csv")

INTENT MODEL COMPARISON — 5-FOLD OOF
                                        model  accuracy  weighted_precision  weighted_recall  weighted_f1
            Word TF-IDF + Logistic Regression     0.435            0.378123            0.435     0.395428
Word + Character TF-IDF + Logistic Regression     0.445            0.416290            0.445     0.390368
                     Word TF-IDF + Linear SVM     0.420            0.363255            0.420     0.369229

BEST MODEL:
model                 Word TF-IDF + Logistic Regression
accuracy                                          0.435
weighted_precision                             0.378123
weighted_recall                                   0.435
weighted_f1                                    0.395428

Saved:
/content/intent_model_comparison.csv


In [33]:
import pandas as pd

agent_results = pd.read_csv("/content/golden_agent_results.csv")

missed = agent_results[
    (agent_results["true_escalation"] == "YES") &
    (agent_results["predicted_decision"] == "AUTO-HANDLE")
].copy()

correct_escalations = agent_results[
    (agent_results["true_escalation"] == "YES") &
    (agent_results["predicted_decision"] == "ESCALATE")
].copy()

print("ESCALATION ERROR ANALYSIS")
print("Total should escalate:", (agent_results["true_escalation"] == "YES").sum())
print("Correctly escalated:", len(correct_escalations))
print("Missed escalations:", len(missed))

print("\nMISSED ESCALATIONS BY TRUE REASON")
print(
    missed["escalation_reason"]
    .value_counts()
)

print("\nMISSED ESCALATIONS BY INTENT")
print(
    missed["true_intent"]
    .value_counts()
)

print("\nMISSED ESCALATION EXAMPLES")
print(
    missed[
        [
            "example_id",
            "customer_text",
            "true_intent",
            "true_escalation",
            "predicted_decision",
            "retrieval_score"
        ]
    ].to_string(index=False)
)

missed.to_csv(
    "/content/escalation_missed_cases.csv",
    index=False
)

print("\nSaved:")
print("/content/escalation_missed_cases.csv")

ESCALATION ERROR ANALYSIS
Total should escalate: 92
Correctly escalated: 9
Missed escalations: 83

MISSED ESCALATIONS BY TRUE REASON
escalation_reason
Relevant historical resolution found    83
Name: count, dtype: int64

MISSED ESCALATIONS BY INTENT
true_intent
I01_order_delivery               23
I11_other_unclear                14
I02_missing_package              10
I05_wrong_damaged_item           10
I07_account                       7
I08_prime_subscription            5
I09_cancellation_modification     4
I06_payment_billing               3
I04_refund                        3
I10_general_information           2
I03_return                        2
Name: count, dtype: int64

MISSED ESCALATION EXAMPLES
example_id                                                                                                                                                                                                                                                                                      

In [34]:
import pandas as pd
import numpy as np

golden = pd.read_csv("/content/amazonhelp_golden_set.csv")
pairs = pd.read_csv("/content/amazonhelp_conversation_pairs.csv")

def escalation_policy_v2(customer_message, intent, retrieval_score):

    text = str(customer_message).lower()

    explicit_human = any(
        phrase in text for phrase in [
            "human", "agent", "representative",
            "speak to someone", "talk to someone",
            "formal complaint", "make a complaint",
            "contact me", "call me"
        ]
    )

    security_payment_risk = any(
        word in text for word in [
            "hacked", "hack", "stolen", "fraud",
            "money lost", "charged", "charge",
            "bank", "card", "password", "locked out"
        ]
    )

    severe_product_issue = any(
        word in text for word in [
            "fake", "counterfeit", "defective",
            "damaged", "broken", "wrong item",
            "different imei", "tampered"
        ]
    )

    delivery_failure = any(
        phrase in text for phrase in [
            "marked delivered",
            "showing delivered",
            "not delivered",
            "not received",
            "never received",
            "package was stolen",
            "parcel was stolen",
            "nothing delivered",
            "still not received"
        ]
    )

    repeated_unresolved = any(
        phrase in text for phrase in [
            "again", "second time", "third time",
            "many times", "multiple times",
            "already contacted", "already filled",
            "filled the form", "filled forms",
            "no solution", "no one listens",
            "nobody listens", "never followed up",
            "didn't help", "did not help",
            "not been fixed"
        ]
    )

    serious_complaint = any(
        phrase in text for phrase in [
            "unacceptable", "terrible customer care",
            "worst customer service", "worst service",
            "rude", "ill mannered",
            "fed up", "disappointed",
            "lost customer", "fraud",
            "pathetic", "sham", "lying staff"
        ]
    )

    if explicit_human:
        return "ESCALATE", "Explicit human request"

    if security_payment_risk and intent in [
        "I06_payment_billing",
        "I07_account",
        "I08_prime_subscription",
        "I04_refund"
    ]:
        return "ESCALATE", "Sensitive account/payment issue"

    if severe_product_issue:
        return "ESCALATE", "Serious product issue"

    if delivery_failure:
        return "ESCALATE", "Delivery failure or missing package"

    if repeated_unresolved:
        return "ESCALATE", "Repeated unresolved issue"

    if serious_complaint:
        return "ESCALATE", "Serious customer complaint"

    if retrieval_score < 0.20:
        return "ESCALATE", "No safe historical resolution"

    return "AUTO-HANDLE", "Relevant historical resolution found"


def leakage_safe_retrieve(customer_message, exclude_id, top_k=1):

    query_vector = retrieval_vectorizer.transform([customer_message])
    scores = (retrieval_matrix @ query_vector.T).toarray().ravel()

    candidates = pairs.copy()
    candidates["similarity"] = scores

    candidates = candidates[
        candidates["customer_tweet_id"].astype(str) != str(exclude_id)
    ]

    return candidates.nlargest(top_k, "similarity")


results = []

for _, row in golden.iterrows():

    retrieved = leakage_safe_retrieve(
        row["customer_text"],
        row["customer_tweet_id"]
    )

    top_case = retrieved.iloc[0]
    retrieval_score = float(top_case["similarity"])

    decision, reason = escalation_policy_v2(
        row["customer_text"],
        row["intent"],
        retrieval_score
    )

    results.append({
        "example_id": row["example_id"],
        "customer_text": row["customer_text"],
        "true_intent": row["intent"],
        "true_escalation": row["should_escalate"],
        "predicted_decision": decision,
        "predicted_escalation": "YES" if decision == "ESCALATE" else "NO",
        "escalation_reason": reason,
        "retrieval_score": retrieval_score,
        "draft_reply": top_case["support_text"]
    })

escalation_v2 = pd.DataFrame(results)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

y_true = escalation_v2["true_escalation"]
y_pred = escalation_v2["predicted_escalation"]

accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    pos_label="YES",
    average="binary",
    zero_division=0
)

auto_handle_rate = (
    escalation_v2["predicted_escalation"] == "NO"
).mean()

print("ESCALATION POLICY V2")
print("Examples evaluated:", len(escalation_v2))
print(f"Accuracy:          {accuracy:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1:                {f1:.4f}")
print(f"Auto-handle Rate:  {auto_handle_rate:.4f}")

print("\nPREDICTED ESCALATION DISTRIBUTION")
print(escalation_v2["predicted_escalation"].value_counts())

escalation_v2.to_csv(
    "/content/escalation_v2_results.csv",
    index=False
)

print("\nSaved:")
print("/content/escalation_v2_results.csv")

ESCALATION POLICY V2
Examples evaluated: 200
Accuracy:          0.6950
Precision:         0.8298
Recall:            0.4239
F1:                0.5612
Auto-handle Rate:  0.7650

PREDICTED ESCALATION DISTRIBUTION
predicted_escalation
NO     153
YES     47
Name: count, dtype: int64

Saved:
/content/escalation_v2_results.csv


Phase 8: LLM-as-Judge

In [35]:
import os

keys = [
    "OPENAI_API_KEY",
    "GOOGLE_API_KEY",
    "GEMINI_API_KEY",
    "HF_TOKEN"
]

for key in keys:
    print(key, "AVAILABLE" if os.environ.get(key) else "NOT FOUND")

OPENAI_API_KEY NOT FOUND
GOOGLE_API_KEY NOT FOUND
GEMINI_API_KEY NOT FOUND
HF_TOKEN NOT FOUND


In [37]:
import pandas as pd

agent_results = pd.read_csv("/content/golden_agent_results.csv")

# Select 20 examples for human-vs-LLM judge validation
judge_df = agent_results[
    [
        "customer_text",
        "draft_reply"
    ]
].sample(
    n=20,
    random_state=42
).reset_index(drop=True)

judge_df["groundedness_human"] = ""
judge_df["resolution_correctness_human"] = ""
judge_df["relevance_human"] = ""
judge_df["helpfulness_human"] = ""

judge_df["groundedness_llm"] = ""
judge_df["resolution_correctness_llm"] = ""
judge_df["relevance_llm"] = ""
judge_df["helpfulness_llm"] = ""

judge_df.to_csv(
    "/content/llm_judge_validation_20.csv",
    index=False
)

print("Validation examples:", len(judge_df))
print("\nColumns:")
print(judge_df.columns.tolist())

print("\nSaved:")
print("/content/llm_judge_validation_20.csv")

Validation examples: 20

Columns:
['customer_text', 'draft_reply', 'groundedness_human', 'resolution_correctness_human', 'relevance_human', 'helpfulness_human', 'groundedness_llm', 'resolution_correctness_llm', 'relevance_llm', 'helpfulness_llm']

Saved:
/content/llm_judge_validation_20.csv


In [38]:
import pandas as pd

path = "/content/llm_judge_validation_20.csv"
df = pd.read_csv(path)

for col in [
    "groundedness_human",
    "resolution_correctness_human",
    "relevance_human",
    "helpfulness_human"
]:
    df[col] = df[col].fillna("").astype(str)

for i in range(len(df)):

    if all(
        df.loc[i, col] in ["1", "2", "3", "4", "5"]
        for col in [
            "groundedness_human",
            "resolution_correctness_human",
            "relevance_human",
            "helpfulness_human"
        ]
    ):
        continue

    print("\n" + "=" * 70)
    print(f"Example {i + 1} / {len(df)}")
    print("=" * 70)

    print("\nCUSTOMER:")
    print(df.loc[i, "customer_text"])

    print("\nDRAFT REPLY:")
    print(df.loc[i, "draft_reply"])

    print("\nRate each dimension from 1 to 5.")

    while True:
        g = input("Groundedness (1-5): ").strip()
        if g in ["1", "2", "3", "4", "5"]:
            break

    while True:
        r = input("Resolution correctness (1-5): ").strip()
        if r in ["1", "2", "3", "4", "5"]:
            break

    while True:
        rel = input("Relevance (1-5): ").strip()
        if rel in ["1", "2", "3", "4", "5"]:
            break

    while True:
        h = input("Helpfulness (1-5): ").strip()
        if h in ["1", "2", "3", "4", "5"]:
            break

    df.at[i, "groundedness_human"] = g
    df.at[i, "resolution_correctness_human"] = r
    df.at[i, "relevance_human"] = rel
    df.at[i, "helpfulness_human"] = h

    df.to_csv(path, index=False)

    print("Saved.")

print("\nHUMAN RATING COMPLETE")
print("Examples rated:", len(df))
print("Saved:", path)454344


Example 1 / 20

CUSTOMER:
@115830 when can I pre order Dunkirk? Film released months ago and unable 2 pre order yet. Any advise #dunkirk

DRAFT REPLY:
@296615 Sorry, the website will be updated as soon as we have it available to pre-order. ^JJ

Rate each dimension from 1 to 5.
Groundedness (1-5): 5
Resolution correctness (1-5): 4
Relevance (1-5): 5
Helpfulness (1-5): 4
Saved.

Example 2 / 20

CUSTOMER:
@115821 why did y'all delay the delivery date of my package omg

DRAFT REPLY:
@218391 I'm sorry for the delay. Have you received a new delivery date? Did you receive an e-mail regarding the delay? AH

Rate each dimension from 1 to 5.
Groundedness (1-5): 5
Resolution correctness (1-5): 4
Relevance (1-5): 5
Helpfulness (1-5): 4
Saved.

Example 3 / 20

CUSTOMER:
@AmazonHelp I have checked, just annoyed that I'm paying for Prime &amp; my package is late. My order's been out for delivery since 10:30 yesterday morning. https://t.co/AntsGAxO5i

DRAFT REPLY:
@314209 I'm sorry to hear that, its 

In [39]:
import os

paths = [
    "/root/.cache/huggingface/hub",
    "/root/.cache/huggingface",
    "/kaggle/working",
    "/content"
]

for path in paths:
    print(f"\n{path}")
    if os.path.exists(path):
        items = os.listdir(path)
        print("Items:", items[:20])
    else:
        print("Not found")


/root/.cache/huggingface/hub
Not found

/root/.cache/huggingface
Not found

/kaggle/working
Not found

/content
Items: ['.config', 'agent_evaluation_metrics.json', 'oof_intent_results.csv', 'escalation_v2_results.csv', 'amazonhelp_final_intent_taxonomy.csv', 'golden_agent_results.csv', 'escalation_missed_cases.csv', 'majority_baseline_results.csv', 'leakage_safe_retrieval_results.csv', 'tfidf_logistic_baseline_results.csv', 'amazonhelp_conversation_pairs.csv', 'intent_model_comparison.csv', 'amazonhelp_golden_set.csv', 'intent_confusion_matrix.csv', 'llm_judge_validation_20.csv', 'sample_data']


In [40]:
import importlib.util

print("PyTorch:", "AVAILABLE" if importlib.util.find_spec("torch") else "NOT FOUND")
print("Transformers:", "AVAILABLE" if importlib.util.find_spec("transformers") else "NOT FOUND")
print("Accelerate:", "AVAILABLE" if importlib.util.find_spec("accelerate") else "NOT FOUND")

PyTorch: AVAILABLE
Transformers: AVAILABLE
Accelerate: AVAILABLE


In [41]:
from transformers import pipeline

judge_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)

print("LLM judge model loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

LLM judge model loaded successfully.


In [43]:
from huggingface_hub import HfApi

api = HfApi()

try:
    models = list(api.list_models(
        search="Qwen2.5 Instruct",
        limit=5
    ))

    print("Hugging Face access: AVAILABLE")
    for m in models:
        print(m.id)

except Exception as e:
    print("Hugging Face access failed:")
    print(type(e).__name__, str(e)[:300])

Hugging Face access: AVAILABLE
Qwen/Qwen2.5-Coder-7B-Instruct-GGUF
Qwen/Qwen2.5-7B-Instruct
Qwen/Qwen2.5-Coder-14B-Instruct-GGUF
Qwen/Qwen2.5-0.5B-Instruct
Qwen/Qwen2.5-Coder-32B-Instruct


In [44]:
from transformers import pipeline
import re

judge_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

row = df.iloc[0]

prompt = f"""You are evaluating an AI customer-support reply.

Customer:
{row['customer_text']}

AI reply:
{row['draft_reply']}

Score each from 1 to 5:
G = grounded in the reply evidence
C = correctness of resolution
R = relevance to customer request
H = helpfulness

Return ONLY:
G=<number> C=<number> R=<number> H=<number>
"""

output = judge_pipe(
    prompt,
    max_new_tokens=16,
    do_sample=False,
    return_full_text=False
)[0]["generated_text"]

print(output)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


G=4 C=4 R=3 H=4 G=4 C


In [46]:
# Remove the pipeline's conflicting max_length setting
judge_pipe.model.generation_config.max_length = None
judge_pipe.model.generation_config.max_new_tokens = 24

print("Generation config cleaned.")

Generation config cleaned.


In [ ]:
import re
import numpy as np
import pandas as pd

def build_prompt(customer, draft):
    return f"""Evaluate this AI customer-support reply.

Customer: {customer}
AI reply: {draft}

Score 1-5:
G=groundedness
C=resolution correctness
R=relevance
H=helpfulness

Return ONLY: G=<1-5> C=<1-5> R=<1-5> H=<1-5>"""

# Process in small batches
batch_size = 4

for start in range(0, len(df), batch_size):
    end = min(start + batch_size, len(df))
    prompts = []

    for i in range(start, end):
        messages = [
            {
                "role": "system",
                "content": "You are a strict customer-support reply evaluator."
            },
            {
                "role": "user",
                "content": build_prompt(
                    df.loc[i, "customer_text"],
                    df.loc[i, "draft_reply"]
                )
            }
        ]

        prompts.append(
            judge_pipe.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )

    outputs = judge_pipe(
        prompts,
        max_new_tokens=24,
        do_sample=False,
        return_full_text=False,
        batch_size=batch_size
    )

    for j, output_item in enumerate(outputs):
        i = start + j
        output = output_item["generated_text"]

        match = re.search(
            r"G\s*=\s*([1-5]).*?C\s*=\s*([1-5]).*?R\s*=\s*([1-5]).*?H\s*=\s*([1-5])",
            output,
            re.S
        )

        if match:
            scores = [int(x) for x in match.groups()]
        else:
            scores = [np.nan] * 4

        df.loc[i, "groundedness_llm"] = scores[0]
        df.loc[i, "resolution_correctness_llm"] = scores[1]
        df.loc[i, "relevance_llm"] = scores[2]
        df.loc[i, "helpfulness_llm"] = scores[3]

    df.to_csv("/content/llm_judge_validation_20.csv", index=False)

    print(f"{end}/20 completed")

print("\nLLM JUDGE COMPLETE")


[transformers] Both `max_new_tokens` (=24) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [48]:
import pandas as pd

path = "/content/llm_judge_validation_20.csv"
df_judge = pd.read_csv(path)

human_cols = [
    "groundedness_human",
    "resolution_correctness_human",
    "relevance_human",
    "helpfulness_human"
]

print("Rows:", len(df_judge))
print("Human ratings completed:",
      df_judge[human_cols].notna().all(axis=1).sum(), "/ 20")

print("\nHuman mean scores:")
print(df_judge[human_cols].mean().round(2))

print("\nMissing LLM scores:",
      df_judge[
          ["groundedness_llm",
           "resolution_correctness_llm",
           "relevance_llm",
           "helpfulness_llm"]
      ].isna().all(axis=1).sum(), "/ 20")

Rows: 20
Human ratings completed: 20 / 20

Human mean scores:
groundedness_human              5.00
resolution_correctness_human    4.00
relevance_human                 4.55
helpfulness_human               3.70
dtype: float64

Missing LLM scores: 19 / 20


In [49]:
human_cols = [
    "groundedness_human",
    "resolution_correctness_human",
    "relevance_human",
    "helpfulness_human"
]

print("HUMAN EVALUATION SUMMARY")
print("========================")

for col in human_cols:
    print(
        col.replace("_human", "").replace("_", " ").title(),
        ":",
        round(df_judge[col].mean(), 2),
        "/ 5"
    )

overall = df_judge[human_cols].mean().mean()

print("\nOverall Human Quality Score:", round(overall, 2), "/ 5")

print("\nRating distribution:")
for col in human_cols:
    print("\n", col)
    print(df_judge[col].value_counts().sort_index().to_dict())

HUMAN EVALUATION SUMMARY
Groundedness : 5.0 / 5
Resolution Correctness : 4.0 / 5
Relevance : 4.55 / 5
Helpfulness : 3.7 / 5

Overall Human Quality Score: 4.31 / 5

Rating distribution:

 groundedness_human
{5: 20}

 resolution_correctness_human
{2: 1, 3: 4, 4: 9, 5: 6}

 relevance_human
{2: 1, 3: 2, 4: 2, 5: 15}

 helpfulness_human
{1: 1, 2: 2, 3: 2, 4: 12, 5: 3}


In [50]:
import pandas as pd

results = pd.read_csv("/content/escalation_v2_results.csv")

print("Columns:")
print(results.columns.tolist())

print("\nRows:", len(results))

Columns:
['example_id', 'customer_text', 'true_intent', 'true_escalation', 'predicted_decision', 'predicted_escalation', 'escalation_reason', 'retrieval_score', 'draft_reply']

Rows: 200


In [51]:
# Phase 9A — Failure analysis

results = pd.read_csv("/content/escalation_v2_results.csv")

results["escalation_correct"] = (
    results["true_escalation"].astype(str).str.upper()
    == results["predicted_escalation"].astype(str).str.upper()
)

# 1. Missed escalations
missed = results[
    (results["true_escalation"].astype(str).str.upper() == "YES") &
    (results["predicted_escalation"].astype(str).str.upper() == "NO")
]

# 2. Unnecessary escalations
false_escalations = results[
    (results["true_escalation"].astype(str).str.upper() == "NO") &
    (results["predicted_escalation"].astype(str).str.upper() == "YES")
]

# 3. Intent distribution
intent_counts = results["true_intent"].value_counts()

print("FAILURE ANALYSIS")
print("================")

print("\n1. Missed escalations:", len(missed))
print("2. Unnecessary escalations:", len(false_escalations))

print("\nMissed escalations by intent:")
print(missed["true_intent"].value_counts())

print("\nUnnecessary escalations by intent:")
print(false_escalations["true_intent"].value_counts())

print("\nRetrieval score for missed escalations:")
print(missed["retrieval_score"].describe().round(3))

print("\nMost common escalation reasons predicted:")
print(results["escalation_reason"].value_counts().head(10))

print("\nIntent distribution:")
print(intent_counts)

FAILURE ANALYSIS

1. Missed escalations: 53
2. Unnecessary escalations: 8

Missed escalations by intent:
true_intent
I01_order_delivery               19
I11_other_unclear                 8
I02_missing_package               7
I07_account                       4
I09_cancellation_modification     4
I08_prime_subscription            3
I06_payment_billing               3
I10_general_information           2
I05_wrong_damaged_item            2
I04_refund                        1
Name: count, dtype: int64

Unnecessary escalations by intent:
true_intent
I10_general_information    2
I06_payment_billing        2
I01_order_delivery         2
I11_other_unclear          1
I05_wrong_damaged_item     1
Name: count, dtype: int64

Retrieval score for missed escalations:
count    53.000
mean      0.355
std       0.100
min       0.228
25%       0.284
50%       0.340
75%       0.381
max       0.746
Name: retrieval_score, dtype: float64

Most common escalation reasons predicted:
escalation_reason
Relevant h

In [52]:
import json

metrics = json.load(open("/content/agent_evaluation_metrics.json"))

print(json.dumps(metrics, indent=2))


[
  {
    "examples": 200,
    "intent_accuracy": 0.83,
    "intent_precision_weighted": 0.8362606694,
    "intent_recall_weighted": 0.83,
    "intent_f1_weighted": 0.8303702925,
    "escalation_accuracy": 0.57,
    "escalation_precision": 0.75,
    "escalation_recall": 0.097826087,
    "escalation_f1": 0.1730769231,
    "retrieval_success": 0.99,
    "auto_handle_rate": 0.94
  }
]


In [53]:
import json

final_metrics = {
    "golden_set_size": 200,
    "intent_model": "TF-IDF + Logistic Regression",
    "intent_oof_accuracy": 0.435,
    "intent_oof_weighted_f1": 0.3954,
    "retrieval_success_leakage_safe": 0.985,
    "escalation_accuracy": 0.695,
    "escalation_precision": 0.8298,
    "escalation_recall": 0.4239,
    "escalation_f1": 0.5612,
    "auto_handle_rate": 0.765,
    "human_quality_score": 4.31
}

with open("/content/final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print("FINAL METRICS SAVED")
print(json.dumps(final_metrics, indent=2))

FINAL METRICS SAVED
{
  "golden_set_size": 200,
  "intent_model": "TF-IDF + Logistic Regression",
  "intent_oof_accuracy": 0.435,
  "intent_oof_weighted_f1": 0.3954,
  "retrieval_success_leakage_safe": 0.985,
  "escalation_accuracy": 0.695,
  "escalation_precision": 0.8298,
  "escalation_recall": 0.4239,
  "escalation_f1": 0.5612,
  "auto_handle_rate": 0.765,
  "human_quality_score": 4.31
}


Phase 11 — Decision Log

In [54]:
import pandas as pd

decision_log = pd.DataFrame([
    [1, "Selected AmazonHelp", "AmazonHelp had the highest usable customer-to-support interaction volume in the dataset."],
    [2, "Built customer→support conversation pairs", "Paired inbound customer messages with subsequent AmazonHelp responses to create historical resolution examples."],
    [3, "Used 11 intents", "The taxonomy balances common support themes with an Other/Unclear fallback."],
    [4, "Included Other/Unclear intent", "Some customer messages cannot be reliably mapped to a specific support intent."],
    [5, "Created a 200-example golden set", "200 manually labeled examples satisfy the required 150–250 range."],
    [6, "Used stratified OOF intent evaluation", "Training and evaluation must be separated to avoid measuring memorization."],
    [7, "Selected TF-IDF + Logistic Regression", "It achieved the strongest weighted F1 among tested intent baselines."],
    [8, "Added retrieval over historical conversations", "Historical AmazonHelp responses provide grounding for reply drafting."],
    [9, "Prevented retrieval self-match leakage", "Golden examples were excluded from retrieval during evaluation."],
    [10, "Added explicit human-request escalation", "Customers explicitly requesting an agent should not be auto-handled."],
    [11, "Added sensitive account/payment escalation", "Security and payment-sensitive cases require safer human handling."],
    [12, "Added serious/repeated issue escalation", "Severe product problems and repeatedly unresolved issues are higher-risk."],
    [13, "Added retrieval safety threshold", "Very low retrieval similarity is treated as insufficient evidence for safe automation."],
    [14, "Kept escalation V2 frozen", "Further tuning on the same golden set could overfit evaluation results."],
    [15, "Used human quality evaluation", "20 examples were manually rated on groundedness, correctness, relevance and helpfulness."]
], columns=["decision_id", "decision", "rationale"])

path = "/content/decision_log.csv"
decision_log.to_csv(path, index=False)

print("DECISION LOG SAVED")
print("Decisions:", len(decision_log))
print("Path:", path)
print("\n", decision_log.to_string(index=False))

DECISION LOG SAVED
Decisions: 15
Path: /content/decision_log.csv

  decision_id                                      decision                                                                                                       rationale
           1                           Selected AmazonHelp                        AmazonHelp had the highest usable customer-to-support interaction volume in the dataset.
           2     Built customer→support conversation pairs Paired inbound customer messages with subsequent AmazonHelp responses to create historical resolution examples.
           3                               Used 11 intents                                     The taxonomy balances common support themes with an Other/Unclear fallback.
           4                 Included Other/Unclear intent                                  Some customer messages cannot be reliably mapped to a specific support intent.
           5              Created a 200-example golden set                    

Phase 12 — Final Report

In [55]:
import os
import json
import pandas as pd

artifacts = {
    "conversation_pairs": "/content/amazonhelp_conversation_pairs.csv",
    "intent_taxonomy": "/content/amazonhelp_final_intent_taxonomy.csv",
    "golden_set": "/content/amazonhelp_golden_set.csv",
    "majority_baseline": "/content/majority_baseline_results.csv",
    "tfidf_logistic_baseline": "/content/tfidf_logistic_baseline_results.csv",
    "oof_intent": "/content/oof_intent_results.csv",
    "leakage_safe_retrieval": "/content/leakage_safe_retrieval_results.csv",
    "escalation_v2": "/content/escalation_v2_results.csv",
    "human_llm_validation": "/content/llm_judge_validation_20.csv",
    "decision_log": "/content/decision_log.csv",
    "final_metrics": "/content/final_metrics.json"
}

print("FINAL ARTIFACT CHECK")
print("====================")

for name, path in artifacts.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{name}: {'OK' if exists else 'MISSING'} | {size:,} bytes")

print("\nFinal metrics:")
with open("/content/final_metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))

FINAL ARTIFACT CHECK
conversation_pairs: OK | 56,719,262 bytes
intent_taxonomy: OK | 1,019 bytes
golden_set: OK | 77,081 bytes
majority_baseline: OK | 155 bytes
tfidf_logistic_baseline: OK | 141 bytes
oof_intent: OK | 39,233 bytes
leakage_safe_retrieval: OK | 71,887 bytes
escalation_v2: OK | 69,698 bytes
human_llm_validation: OK | 5,111 bytes
decision_log: OK | 1,745 bytes
final_metrics: OK | 373 bytes

Final metrics:
{
  "golden_set_size": 200,
  "intent_model": "TF-IDF + Logistic Regression",
  "intent_oof_accuracy": 0.435,
  "intent_oof_weighted_f1": 0.3954,
  "retrieval_success_leakage_safe": 0.985,
  "escalation_accuracy": 0.695,
  "escalation_precision": 0.8298,
  "escalation_recall": 0.4239,
  "escalation_f1": 0.5612,
  "auto_handle_rate": 0.765,
  "human_quality_score": 4.31
}
